# Chủ đề 5 — OCR hóa đơn bán lẻ
Chạy lần lượt các ô. Dùng Python 3.10+ và Tesseract eng. Xem README trước khi chạy. Không chỉnh luật sau khi đọc test.

In [1]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT / "sroie_project"
os.chdir(ROOT)
assert (ROOT / "app.py").exists()
sys.path.insert(0,str(ROOT))


## Dữ liệu và chia tập
Tập con 300 ảnh của bản metatatt/sorie; hold-out nội bộ 200 ảnh, 80 train và 20 validation. Đây không phải test chính thức ICDAR. Manifest giữ nguyên seed 42.

In [2]:
from src.data_loader import get_splits
splits = get_splits()
{k: len(v) for k,v in splits.items()}

{'train': 80, 'val': 20, 'test': 200}


In [3]:
subprocess.run([sys.executable,"scripts/01_explore_data.py"],check=True)

CompletedProcess(args=['/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python', 'scripts/01_explore_data.py'], returncode=0)


## Baseline và cải tiến
Baseline lấy dòng đầu, hai dòng địa chỉ, regex ngày và tiền lớn nhất. Improved chấm điểm tên đơn vị, ưu tiên ngày có DATE và total có từ khóa. So sánh 2 biến thể ảnh × 2 bộ luật.

In [4]:
from src.extractors import baseline_extract, improved_extract
lines=["ABC SDN BHD","DATE 25/12/2018","TOTAL 18.05","CASH 50.00"]
baseline_extract(lines), improved_extract(lines)

({'company': 'ABC SDN BHD', 'date': '25/12/2018', 'address': 'DATE 25/12/2018 TOTAL 18.05', 'total': '50.00'}, {'company': 'ABC SDN BHD', 'date': '25/12/2018', 'address': None, 'total': '18.05'})


In [5]:
subprocess.run([sys.executable,"scripts/run_experiment.py"],check=True)

CompletedProcess(args=['/opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python', 'scripts/run_experiment.py'], returncode=0)


In [6]:
import pandas as pd
metrics=pd.read_csv("outputs/experiment/metrics.csv")
metrics

  split             config    n  ...  EM_address  EM_date  EM_total
0   val     basic/baseline   20  ...       0.050    0.550     0.400
1   val     basic/improved   20  ...       0.150    0.800     0.600
2   val  improved/baseline   20  ...       0.000    0.400     0.550
3   val  improved/improved   20  ...       0.050    0.500     0.500
4  test     basic/baseline  200  ...       0.070    0.645     0.355
5  test     basic/improved  200  ...       0.210    0.745     0.555
6  test  improved/baseline  200  ...       0.010    0.665     0.330
7  test  improved/improved  200  ...       0.095    0.760     0.485

[8 rows x 11 columns]


## Phân tích lỗi
error_type là chẩn đoán heuristic; xem ảnh gốc để xác nhận. Không thay đổi nhãn để tăng điểm.

In [7]:
errors=pd.read_csv("outputs/experiment/per_field.csv")
errors[(errors["split"]=="test") & (~errors["correct"])].head(20)

     id split  ... correct        error_type
320   5  test  ...   False  extraction_error
321   5  test  ...   False         ocr_error
323   5  test  ...   False         ocr_error
325   5  test  ...   False         ocr_error
327   5  test  ...   False         ocr_error
328   5  test  ...   False         ocr_error
329   5  test  ...   False         ocr_error
331   5  test  ...   False         ocr_error
332   5  test  ...   False         ocr_error
333   5  test  ...   False         ocr_error
335   5  test  ...   False         ocr_error
336   8  test  ...   False         ocr_error
337   8  test  ...   False         ocr_error
338   8  test  ...   False         ocr_error
339   8  test  ...   False         ocr_error
340   8  test  ...   False         ocr_error
341   8  test  ...   False         ocr_error
342   8  test  ...   False         ocr_error
343   8  test  ...   False         ocr_error
344   8  test  ...   False         ocr_error

[20 rows x 8 columns]


## Demo
Chạy trong terminal: `python -m streamlit run app.py`. Chọn ảnh, engine, tiền xử lý và luật; đọc OCR, JSON và giá trị chuẩn hóa.